# Train Hate Speech Detection on Colab

This notebook clones or updates the project from GitHub, prepares data, then calls the production Python modules in `src/`.

Default branch is `main`. Change `BRANCH` in the first code cell if you want to train from another branch, for example `refactor`.

In [1]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/thong7d/hate-speech-detection.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/hate-speech-detection")

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").exists():
        raise RuntimeError(f"{PROJECT_DIR} exists but is not a git repository. Remove it or choose another PROJECT_DIR.")
    os.chdir(PROJECT_DIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)

print("Project directory:", Path.cwd())
subprocess.run(["git", "branch", "--show-current"], check=True)

Project directory: /content/hate-speech-detection


CompletedProcess(args=['git', 'branch', '--show-current'], returncode=0)

In [2]:
from pathlib import Path

skip_packages = ("underthesea", "gradio", "pyngrok", "google-generativeai")
source = Path("requirements.txt")
target = Path("requirements-colab.txt")
lines = []
for line in source.read_text(encoding="utf-8").splitlines():
    stripped = line.strip()
    if stripped and not stripped.startswith(skip_packages):
        lines.append(line)
target.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(target.read_text(encoding="utf-8"))

transformers==4.40.2
datasets==2.19.2
torch>=2.1.0
accelerate==0.29.3
scikit-learn==1.4.2
pandas==2.2.2
pyarrow==16.0.0
requests==2.32.2
PyYAML==6.0.2
matplotlib==3.9.0
seaborn==0.13.2
langdetect==1.0.9
fastapi==0.111.0
uvicorn==0.29.0
omegaconf==2.3.0
sentencepiece==0.2.0
protobuf==4.25.3



In [3]:
!python -m pip install -U pip setuptools wheel
# Gỡ bỏ TensorFlow triệt để để ngăn transformers gọi các module gây lỗi protobuf
!python -m pip uninstall -y tensorflow
# Không can thiệp vào protobuf, để Colab dùng bản mặc định
!python -m pip install -r requirements-colab.txt
!python -m pip install huggingface_hub sentencepiece
!python -m pip uninstall -y peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.0 MB/s  0:00

In [4]:
from pathlib import Path

from src.data.download import download_and_extract
from src.data.preprocessing import process_split

raw_paths = {
    "train": Path("data/raw/vihsd/train.csv"),
    "dev": Path("data/raw/vihsd/dev.csv"),
    "test": Path("data/raw/vihsd/test.csv"),
}
processed_paths = {
    "train": Path("data/processed/train.parquet"),
    "dev": Path("data/processed/dev.parquet"),
    "test": Path("data/processed/test.parquet"),
}

if not all(path.exists() for path in raw_paths.values()):
    download_and_extract("configs/paths.yaml")

for split in ("train", "dev", "test"):
    df = process_split(raw_paths[split], processed_paths[split], use_word_segmentation=False)
    print(f"{split}: {len(df)} rows -> {processed_paths[split]}")

Download successful.
Extracting files...
[OK] Extracted: test.csv -> /content/hate-speech-detection/data/raw/vihsd/test.csv
[OK] Extracted: dev.csv -> /content/hate-speech-detection/data/raw/vihsd/dev.csv
[OK] Extracted: train.csv -> /content/hate-speech-detection/data/raw/vihsd/train.csv
[SUCCESS] Phase 1: Data Acquisition completed successfully.
train: 23833 rows -> data/processed/train.parquet
dev: 2639 rows -> data/processed/dev.parquet
test: 6613 rows -> data/processed/test.parquet


In [5]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
else:
    print("Khong du du lieu de xac minh HF_TOKEN hoac HF_TOKEN chua duoc cau hinh")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
!python -c "import torch, transformers, pandas, pyarrow, sklearn, src; print('OK')"

OK


In [7]:
from huggingface_hub import snapshot_download
import os
import shutil

# Tải trạng thái checkpoint của Epoch 6 từ Hub về máy ảo Colab
snapshot_download(
    repo_id="thong7d/vihsd-xlmr-hate-speech",
    allow_patterns="last-checkpoint/*",
    local_dir="./hf_download"
)

# Cấu trúc lại thành thư mục checkpoint chuẩn cho Trainer nhận diện
output_dir = "outputs"  # Thay đổi bằng giá trị training.output_dir trong file yaml của bạn
checkpoint_dir = os.path.join(output_dir, "checkpoint-10623") # Sử dụng step tương ứng với Epoch 6
os.makedirs(checkpoint_dir, exist_ok=True)

src_dir = "./hf_download/last-checkpoint"
for filename in os.listdir(src_dir):
    shutil.move(os.path.join(src_dir, filename), os.path.join(checkpoint_dir, filename))

print("Khôi phục checkpoint cục bộ thành công tại:", checkpoint_dir)

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

last-checkpoint/sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

last-checkpoint/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

last-checkpoint/scheduler.pt:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

last-checkpoint/training_args.bin:   0%|          | 0.00/5.58k [00:00<?, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

Khôi phục checkpoint cục bộ thành công tại: outputs/checkpoint-10623


In [8]:
# Tạo thư mục outputs bên trong kho lưu trữ
!mkdir -p /content/hate-speech-detection/outputs

# Di chuyển thư mục checkpoint vào đúng vị trí hệ thống tìm kiếm
!mv /content/outputs/checkpoint-10623 /content/hate-speech-detection/outputs/

# Kiểm tra xác nhận sự tồn tại của các tệp tin cấu trúc
!ls -la /content/hate-speech-detection/outputs/checkpoint-10623

mv: cannot stat '/content/outputs/checkpoint-10623': No such file or directory
total 3289460
drwxr-xr-x 2 root root       4096 May 31 10:47 .
drwxr-xr-x 3 root root       4096 May 31 10:47 ..
-rw-r--r-- 1 root root        851 May 31 10:47 config.json
-rw-r--r-- 1 root root 1115350164 May 31 10:47 model.safetensors
-rw-r--r-- 1 root root 2230828363 May 31 10:47 optimizer.pt
-rw-r--r-- 1 root root      14645 May 31 10:47 rng_state.pth
-rw-r--r-- 1 root root       1977 May 31 10:47 scheduler.pt
-rw-r--r-- 1 root root    5069051 May 31 10:47 sentencepiece.bpe.model
-rw-r--r-- 1 root root        280 May 31 10:47 special_tokens_map.json
-rw-r--r-- 1 root root       1147 May 31 10:47 tokenizer_config.json
-rw-r--r-- 1 root root   17082999 May 31 10:47 tokenizer.json
-rw-r--r-- 1 root root      11385 May 31 10:47 trainer_state.json
-rw-r--r-- 1 root root       5585 May 31 10:47 training_args.bin


In [9]:
!pwd

/content/hate-speech-detection


In [10]:
import yaml
import os
import shutil
from pathlib import Path

# Đọc cấu hình thực tế từ tệp train.yaml
config_path = "/content/hate-speech-detection/configs/train.yaml"
if not os.path.exists(config_path):
    config_path = "configs/train.yaml"

with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

output_dir_raw = config["training"]["output_dir"]
project_root = Path("/content/hate-speech-detection")

# Xác định đường dẫn tuyệt đối chính xác mà Trainer sẽ quét
if os.path.isabs(output_dir_raw):
    true_output_dir = Path(output_dir_raw)
else:
    true_output_dir = project_root / output_dir_raw

true_output_dir.mkdir(parents=True, exist_ok=True)
target_checkpoint = true_output_dir / "checkpoint-10623"

# Các vị trí nguồn có khả năng đang lưu trữ checkpoint
sources = [
    Path("/content/outputs/checkpoint-10623"),
    Path("/content/hate-speech-detection/outputs/checkpoint-10623"),
    Path("./outputs/checkpoint-10623")
]

moved = False
for src in sources:
    if src.exists() and src.resolve() != target_checkpoint.resolve():
        if target_checkpoint.exists():
            shutil.rmtree(target_checkpoint)
        shutil.copytree(src, target_checkpoint)
        print(f"[STATUS] Synchronized checkpoint from {src} to correct location: {target_checkpoint}")
        moved = True
        break

if not moved and target_checkpoint.exists():
    print(f"[STATUS] Checkpoint is already correctly positioned at: {target_checkpoint}")
elif not moved:
    print("[ERROR] Could not find 'checkpoint-10623' in any known staging directories.")

[STATUS] Synchronized checkpoint from /content/hate-speech-detection/outputs/checkpoint-10623 to correct location: /content/hate-speech-detection/artifacts/hate_speech_model/checkpoint/checkpoint-10623


In [11]:
!python -m src.training.train --config configs/train.yaml

/content/hate-speech-detection/src/training/train.py:13: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  if hasattr(np, "core") and hasattr(np.core, "multiarray") and hasattr(np.core.multiarray, "_reconstruct"):
/content/hate-speech-detection/src/training/train.py:14: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy

In [15]:
import yaml
import os
from pathlib import Path
from src.training.trainer import train_from_config
from src.utils.config import load_yaml_config

# 1. Nạp tệp cấu hình gốc
config_path = "/content/hate-speech-detection/configs/train.yaml"
if not os.path.exists(config_path):
    config_path = "configs/train.yaml"

cfg = load_yaml_config(config_path)

# 2. Thay đổi tham số để ép Trainer nhận diện tiến trình đã hoàn thành
# Đặt max_steps bằng chính xác mốc step của checkpoint hiện tại
cfg["training"]["max_steps"] = 15931
cfg["training"]["resume_from_checkpoint"] = "/content/hate-speech-detection/artifacts/hate_speech_model/checkpoint/checkpoint-15931"

# Khóa tính năng push_to_hub trong luồng xử lý này để tránh treo mạng lần 2
cfg["training"]["push_to_hub"] = False

# 3. Khởi chạy tiến trình đóng gói
print("[INFO] Kích hoạt luồng trích xuất trọng số tối ưu từ checkpoint-15931...")
train_from_config(cfg)
print("[SUCCESS] Mô hình tối ưu nhất đã được đóng gói thành công tại thư mục artifacts/hate_speech_model/model")

[INFO] Kích hoạt luồng trích xuất trọng số tối ưu từ checkpoint-15931...
[TRAIN] Final training set size: 56648 samples
[TRAIN] Validation set size: 2639 samples
[TRAIN] Label distribution: {2: 20470, 0: 19970, 1: 16208}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Some weights of XLMRobertaTextCNN were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['convs.0.bias', 'convs.0.weight', 'convs.1.bias', 'convs.1.weight', 'convs.2.bias', 'convs.2.weight', 'convs.3.bias', 'convs.3.weight', 'fc.bias', 'fc.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[TRAIN] Phát hiện checkpoint hợp lệ. Khôi phục tiến trình từ: /content/hate-speech-detection/artifacts/hate_speech_model/checkpoint/checkpoint-15931


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,Macro F1,Weighted F1,Precision Clean,Recall Clean,F1 Clean,Precision Offensive,Recall Offensive,F1 Offensive,Precision Hate,Recall Hate,F1 Hate,Critical F1,Critical Recall,Offensive Priority F1,Offensive Priority Recall,Balanced Critical F1,Balanced Critical Recall
9,0.015600,0.518001,0.854500,0.658600,0.652900,0.654600,0.853100,0.923400,0.926800,0.925100,0.475100,0.409500,0.439900,0.577300,0.622200,0.598900,0.519400,0.515900,0.487600,0.473300,0.546500,0.543300


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

  ...ckpoint/training_args.bin: 100%|##########| 5.58kB / 5.58kB            

  ...ckpoint/model.safetensors:   1%|1         | 16.0MB / 1.12GB            

  ...t/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

  ...checkpoint/tokenizer.json:  93%|#########3| 15.9MB / 17.1MB            

[SUCCESS] Mô hình tối ưu nhất đã được đóng gói thành công tại thư mục artifacts/hate_speech_model/model


In [19]:
from huggingface_hub import HfApi

api = HfApi()
print("[INFO] Uploading consolidated optimal model directory to Hugging Face Hub...")

api.upload_folder(
    folder_path="/content/hate-speech-detection/artifacts/hate_speech_model/model",
    repo_id="thong7d/vihsd-xlmr-hate-speech",
    repo_type="model"
)

print("[SUCCESS] Model directly published to https://huggingface.co/thong7d/vihsd-xlmr-hate-speech")

[INFO] Uploading consolidated optimal model directory to Hugging Face Hub...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...l/model/training_args.bin: 100%|##########| 5.58kB / 5.58kB            

  ...odel/model/tokenizer.json:  93%|#########3| 15.9MB / 17.1MB            

  ...l/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

  ...l/model/model.safetensors:   1%|1         | 16.0MB / 1.12GB            

No files have been modified since last commit. Skipping to prevent empty commit.


[SUCCESS] Model directly published to https://huggingface.co/thong7d/vihsd-xlmr-hate-speech


In [16]:
!python -m src.evaluation.evaluate --config configs/train.yaml

/content/hate-speech-detection/src/evaluation/evaluate.py:13: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  if hasattr(np, "core") and hasattr(np.core, "multiarray") and hasattr(np.core.multiarray, "_reconstruct"):
/content/hate-speech-detection/src/evaluation/evaluate.py:14: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usag

In [17]:
push_flag = "--push-to-hub" if os.environ.get("HF_TOKEN") else ""
!python -m src.export.export_model --config configs/train.yaml {push_flag}

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...l/sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<?, ?B/s]


  ...l/model/training_args.bin: 100% 5.58k/5.58k [00:00<?, ?B/s]

  ...l/sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<?, ?B/s]


  ...l/model/training_args.bin: 100% 5.58k/5.58k [00:00<?, ?B/s]



  ...odel/model/tokenizer.json: 100% 17.1M/17.1M [00:00<?, ?B/s]




  ...l/model/model.safetensors:   2% 24.0M/1.12G [00:00<?, ?B/s]

  ...l/sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<?, ?B/s]


  ...l/model/training_args.bin: 100% 5.58k/5.58k [00:00<?, ?B/s]



  ...odel/model/tokenizer.json: 100% 17.1M/17.1M [00:00<?, ?B/s]




Processing Files (3 / 4)      :   4% 46.1M/1.14G [00:00<00:07, 144MB/s,   ???B/s  ]

  ...l/sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<?, ?B/s]


  ...l/model/training_args.bin: 100% 5.58k/5.58k [00:00<?, ?B/s]



  ...odel/model/tokenizer.

In [18]:
!python -m src.evaluation.manual_tests --config configs/model.yaml

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
config.json: 100% 945/945 [00:00<00:00, 5.94MB/s]
tokenizer_config.json: 1.15kB [00:00, 1.97MB/s]
model/sentencepiece.bpe.model: 100% 5.07M/5.07M [00:02<00:00, 2.14MB/s]
model/tokenizer.json: 100% 17.1M/17.1M [00:01<00:00, 14.8MB/s]
special_tokens_map.json: 100% 280/280 [00:00<00:00, 2.01MB/s]
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
model/model.safetensors: 100% 1.12G/1.12G [00:57<00:00, 19.3MB/s]
Manual test rows: 68
Report written to results/manual_test_report.md


Outputs:

- Local artifact: `artifacts/hate_speech_model/`
- Final model: `artifacts/hate_speech_model/model/`
- Hugging Face repo: configured in `configs/train.yaml`